In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pandas as pd
import torch.nn as nn
import optuna
from optuna.pruners import MedianPruner
from optuna.trial import TrialState
import torch
from torch.utils.data import DataLoader
from scripts_proporciones import models_training
from scripts_proporciones.models import MRConvolutionalModel,MRConvolutionalModelHistogram
from scripts_proporciones.create_dataset import CustomImageDataset

# Prueba de detección de esquinas con técnicas de visión por computación

In [ ]:
# Carga de imágenes para usar tecnicas de vision con cv2 
def load_images_cv2(file):  
    img=cv2.imread(file)
    img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

    h=img.shape[0]
    w=img.shape[1]
    if w<h: #Pasamos todas las imagenes a horizontal
        img=cv2.rotate(img,cv2.ROTATE_90_COUNTERCLOCKWISE)
    img=cv2.resize(img,(1920,1080))    
    return img

image_dir="Pruebas"
image_files=[os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith('.jpg')]

dataset=[load_images_cv2(i) for i in image_files]

NameError: name 'os' is not defined

In [ ]:
def four_vertices(contour):
    # Función para buscar una aproximación el contorno que tenga 4 vértices
    for i in np.linspace(0.01,0.1,50):
        aprox=cv2.approxPolyDP(contour,i*cv2.arcLength(contour,True),True)
        if len(aprox)==4:
            return aprox
    return None

In [ ]:
# Cargar imagen
img = dataset[1].copy()
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

# Suavizado (mejorado)
gray = cv2.equalizeHist(gray)
gray = cv2.bilateralFilter(gray, d=9, sigmaColor=75, sigmaSpace=75)

# Detección de bordes
edges = cv2.Canny(gray, 40, 120)

# Cierre morfológico para unir bordes
kernel = np.ones((5,5), np.uint8)
edges_closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)

# Encontrar contornos externos
contours, _ = cv2.findContours(edges_closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours = [c for c in contours if cv2.contourArea(c) > 2000]

# Seleccionar contorno más grande y aplicar hull
contour = max(contours, key=cv2.contourArea)
contour = cv2.convexHull(contour)

# Aproximación adaptativa
approx = four_vertices(contour)

# Dibujar resultados
output = img.copy()
cv2.drawContours(output, [approx], -1, (0,255,0), 3)
for (x, y) in approx.reshape(-1, 2):
    cv2.circle(output, (x, y), 8, (255,0,0), -1)

plt.figure(figsize=(14,6))
plt.subplot(1,3,1); plt.imshow(gray, cmap='gray'); plt.title("Gris (bilateral)")
plt.axis("off")
plt.subplot(1,3,2); plt.imshow(edges_closed, cmap='gray'); plt.title("Bordes (cerrados)")
plt.axis("off")
plt.subplot(1,3,3); plt.imshow(output)
plt.title(f"Esquinas detectadas: {len(approx)}")
plt.axis("off")
plt.tight_layout()
plt.show()

# Mostrar coordenadas
print("Coordenadas de las esquinas detectadas:")
for i, (x, y) in enumerate(approx.reshape(-1, 2)):
    print(f"  Esquina {i+1}: (x={x}, y={y})")


# Red de clasificación y coordenadas con pytorch, búsqueda de hiperparámetros

In [ ]:
# En el script training está el codigo para entrenar el modelo
# En el script pipeline está el codigo para aplicar el modelo a un conjunto de imagenes

In [ ]:
""""!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!"""
### Vamos a hacer la prueba de hiperparametros utilizando optuna ###
from scripts_recorte.corners_training import complete_training_crop

# Primero probamos pocas estapas para simplemente descartar modelos que no sean interesantes
def objective1(trial):
    try:
        # Tamaños de las capas totalmente conectadas que añadimos al modelo de base (diferentes modelos rinden mejor con diferentes tamaños)
        size1 = trial.suggest_int("size1",512,1024,step=128)
        size2 = trial.suggest_int("size2",128,512,step=64)
        # Modelo base a utilizar
        model_name = trial.suggest_categorical("model_name",['ResNet50','EfficientNetV2_small','ConvNeXt_tiny','MobileNet_V3_Large'])
        # Tasa de dropout a emplear en el modelo
        dropout = trial.suggest_float("dropout",0.1,0.4)
        # Learning rates
        lr1 = trial.suggest_float("lr1", 1e-4, 1e-2, log=True)
        # Optimizador a utilizar
        optimizer = trial.suggest_categorical("optimizer",["SGD", "Adam", "AdamW", "RMSprop"])
        # Tamaño de los batches a emplear
        batch_size = trial.suggest_int("batch_size",32,64,step=32)
        
        parameters = {'size1': size1,'size2':size2,'model_name': model_name,'dropout':dropout, 'lr1':lr1,'optimizer':optimizer,'batch_size':batch_size}

        print("Empezando un nuevo intento con los siguientes parámetros:",parameters)

        # Esta función se llama al final de cada época y sirve para podar las ejecuciones menos prometedoras
        def epoch_callback(loss, epoch):
            trial.report(loss, epoch)
            if trial.should_prune():
                print("El trial no era prometedor y va a ser podado")
                raise optuna.exceptions.TrialPruned()

        # Elegimos la gpu si está disponible y si no la cpu
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Cargamos en data frames los conjuntos de datos ya divididos
        train = pd.read_csv("./dataset_dividido/train_crop.csv")
        val = pd.read_csv("./dataset_dividido/val_crop.csv")

        # Creamos los datasets y dataloaders que vamos a usar para entrenar y validar
        train_dataset = CustomImageDataset("./1_photos",train,True)
        val_dataset = CustomImageDataset("./1_photos",val,True)
        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
        val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

        return complete_training_crop(model_name,optimizer,train_dataloader,val_dataloader,1,1,lr1,None,
                                     dropout,size1,size2,patience1=10,max_epochs1=20,fine_tuning=False,device=device,callback=epoch_callback)
    
    # Tratamiento de errores
    except optuna.exceptions.TrialPruned as e:
        raise
    except Exception as e:
        print("El trial falló, se evitarán esos hiperparámetros")
        print(e)
        raise  

# Realizamos la búsqueda de hiperparámetros utilizando optuna
study = optuna.create_study(
    direction="minimize",
    study_name="RECORTE1",
    storage="sqlite:///Hiperparametros_recorte/Primera_prueba.db",
    load_if_exists=True,
)
study.optimize(objective1,n_trials=30)

In [ ]:
# ---------- Estudiamos los hiperparámetros con mejores resultados de la primera prueba y asi reducimos el grupo de búsqueda para la segunda prueba

# Mejores parámetros obtenidos en la primera prueba
study = optuna.load_study(
    study_name="RECORTE1",
    storage="sqlite:///Hiperparametros_recorte/Primera_prueba.db"
)
top_trials = sorted(
    [t for t in study.trials if t.state == TrialState.COMPLETE],
    key=lambda t: t.value
)[:10]
for i, trial in enumerate(top_trials, 1):
    print("Parámetros:", trial.params)

Parámetros: {'size1': 465, 'size2': 55, 'model_name': 'EfficientNetB1', 'dropout': 0.2610365629319875, 'lr1': 0.001044669447696381, 'lr2': 1.056784606385245e-06, 'optimizer': 'Adam', 'imp_coords': 0.8720592764394149, 'imp_corners': 0.20058791927342934, 'loss_name': 'Huber', 'patience': 9}
Parámetros: {'size1': 399, 'size2': 121, 'model_name': 'EfficientNetB1', 'dropout': 0.33433735553441524, 'lr1': 0.0009931986823591605, 'lr2': 1.3235541152210324e-06, 'optimizer': 'Adam', 'imp_coords': 0.8591051573703635, 'imp_corners': 0.2758898523045693, 'loss_name': 'Huber', 'patience': 10}
Parámetros: {'size1': 476, 'size2': 58, 'model_name': 'EfficientNetB1', 'dropout': 0.10357803496335095, 'lr1': 0.0004988576712744592, 'lr2': 1.2514353704971703e-05, 'optimizer': 'Adam', 'imp_coords': 0.8813570366769187, 'imp_corners': 0.2168512845691126, 'loss_name': 'Huber', 'patience': 9}
Parámetros: {'size1': 498, 'size2': 29, 'model_name': 'EfficientNetB1', 'dropout': 0.16915214344700333, 'lr1': 0.00178265976

In [ ]:
# Parámetros elegidos:
"""
size1=550
size2=150
model_name=EfficientNetB3
dropout=0.01

--- Se decide solo entrenar una vez el sistema completo y con los 3 bloques finales del modelo base descongelados

lr3=9e-4
optimizer=Adam
imp_coords=0.9
imp_corners=0.1
loss_module=MAE
bloques=3
"""

# Preprocesado de las etiquetas

In [ ]:
# Cargamos el excel usando pandas y nos quedamos solo con las columnas de las etiquetas, la del nombre de la foto y las de la latitud y longitud
# Además vamos a quitar la palabra cobertura de las columnas de las etiquetas y poner todos los índices en minúscula para una mayor legibilidad.

cols = list(range(27,39))+[42,43]
labels_route = "2_labels/1_GROUNDTRUTH_table_NORGRASS25_clean.xlsx"
df = pd.read_excel(labels_route,usecols=cols)
df.columns = df.columns.str.lower().str.replace("cobertura","").str.strip()
target_cols = ['n. noltei','z. marina','g. vermiculophylla','sedimento','arena','roca','algas verdes','algas pardas','algas rojas','microfitobentos']

# Nos quedamos solo con los registros que tengan alguna de las fotos recortadas en su campo "foto"
# Esto nos ahorra además tener que eliminar los registros que tengan un valor vacío en la columna correspondiente
cropped_photos = [n for n in os.listdir("fotos_recortadas") if n.endswith((".jpg",".jpeg",".png"))]
df = df[df["foto"].isin(cropped_photos)]

# Vamos a pasar las columnas a float para evitar futuros errores
df[target_cols] = df[target_cols].astype(float)

# La columna "punto verificación" tiene muchos valores faltantes, pero solo la hemos cargado para un estudio posterior por lo que vamos 
#   a rellenar estos valores faltantes con 0s sin detenernos mucho
df["punto verificacion"] = df["punto verificacion"].fillna(0)
df

,punto verificacion,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos,foto,latitude,longitude
1,35.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_191501.jpg,43.639455,-8.086088
2,35.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_191354.jpg,43.639434,-8.086167
3,35.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_191329.jpg,43.639482,-8.086189
4,35.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_191305.jpg,43.639489,-8.086118
5,35.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_191241.jpg,43.639475,-8.086132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2446,0.0,50.0,0.0,0.0,20.0,0.0,0.0,30.0,0.0,0.0,0.0,PHOTO_20250728_114615.jpg,42.429020,-8.655178
2447,0.0,40.0,0.0,0.0,50.0,0.0,0.0,10.0,0.0,0.0,0.0,PHOTO_20250728_114413.jpg,42.429072,-8.655183
2448,0.0,0.0,0.0,0.0,92.0,0.0,0.0,5.0,3.0,0.0,0.0,PHOTO_20250728_113013.jpg,42.428860,-8.655435
2449,0.0,50.0,0.0,0.0,30.0,0.0,0.0,20.0,0.0,0.0,0.0,PHOTO_20250728_113931.jpg,42.428815,-8.655325


## Tratamiento de registros duplicados en función del campo foto

In [ ]:
# -- Vamos a estudiar las carácterísticas de los registros que usan por algún motivo la misma foto
dups=df[df.duplicated(subset=["foto"],keep=False)]

# False - coinciden los valores de todos los campos para los registros con la misma foto (básicamente registros idénticos duplicados)
# True  - no coinciden los valores de todos los campos para los registros con la misma foto (un registro es el verdadero y los demás tienen mediciones incorrectas)
complete_dups=(dups.groupby("foto").nunique()[target_cols]!=1).any(axis=1).reset_index(name="dups")
incorrect_dups=complete_dups[complete_dups["dups"]]["foto"]
dups=dups[dups["foto"].isin(incorrect_dups)]
# De momento solo nos preocupamos por aquellos duplicados con diferente porcentaje de cada clase porque de los otros nos basta con quedarnos con uno de ellos

# -- Ahora no nos queda más remedio que ir comprobando para cada conjunto de duplicados la foto original y decidir cúal de los registros se corresponde con ella
# Podríamos eliminar directamente este tipo de duplicados del conjunto de datos y quedarnos solo con los duplicados que coinciden entre sí limpios
#  pero como son pocos vamos a elegir uno del otro tipo de duplicados que se corresponda con la imagen
# (vamos a ignorar de momento que a veces no suman 100 las columnas)
dups


,punto verificacion,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos,foto,latitude,longitude
182,0.0,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,PHOTO_20250521_170652.jpg,43.637018,-8.055258
183,46.0,0.0,0.0,0.0,20.0,80.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_170652.jpg,43.656589,-8.053690
261,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,PHOTO_20250523_205348.jpg,43.493829,-8.164109
262,0.0,0.0,80.0,0.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,PHOTO_20250523_205348.jpg,43.494301,-8.163640
357,0.0,80.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250523_194738.jpg,43.491732,-8.164747
358,0.0,50.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250523_194738.jpg,43.491878,-8.185995
389,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250523_191013.jpg,43.514130,-8.153036
390,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250523_191013.jpg,43.503046,-8.164047
414,0.0,0.0,0.0,0.0,60.0,0.0,0.0,40.0,0.0,0.0,0.0,PHOTO_20250523_184406.jpg,43.511076,-8.154688
415,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250523_184406.jpg,43.513221,-8.156541


In [4]:
""""
Después de analizar las fotos más fácilmente distinguibles llegamos a la conclusión de que el último registro es siempre el correcto en estos casos, por tanto
como tenemos pocas imagenes restantes que sean problemáticas podríamos suponer que en todos los casos la correcta es el segundo registro, esto significaría
seguramente que se cargaron dos imágenes con el mismo nombre pero la segunda reemplazo a la primera y por eso es el último registro el correcto.
Aún asi asumiremos que no siempre se cumplirá y vamos primero a eliminar todos los registros de fotos problemáticas que sea difícil distinguir el correcto, y
luego de los que queden nos quedaremos con el último registro, en este caso se pierden solo 5 y así evitamos datos mal etiquetados en la medida de lo posible
"""
# Lista de los duplicados problemáticos que vamos a eliminar
delete_index=[357,358,821,822,1104,1105,1247,1248,1875,1876]
df=df.drop(index=delete_index)
df=df.drop_duplicates(subset="foto",keep="last")

## Tratamiento de valores faltantes o incoherentes en las columnas objetivo

In [5]:
# Primero hacemos una comprobación rápida para ver si hay valores faltantes en nuestras columnas objetivo
print(df[df[target_cols].isna().any(axis=1)])
# Nos devuelve un empty data frame porque no hay valores faltantes

# Sabemos que nuestras columnas objetivo incluyen el porcentaje de cada clase que hay presente en la imagen
# Por tanto la suma de las columnas para un registro siempre debería sumar 100, vamos a comprobar si hay 
#  registros en los que la suma de más o menos de 100 y estudiaremos el posible por qué y buscaremos una solución
wrong_values=df[df.loc[:,target_cols].sum(axis=1)!=100]
wrong_values

Empty DataFrame
Columns: [punto verificacion, n. noltei, z. marina, g. vermiculophylla, sedimento, arena, roca, algas verdes, algas pardas, algas rojas, microfitobentos, foto, latitude, longitude]
Index: []


,punto verificacion,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos,foto,latitude,longitude
6,0.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,70.0,PHOTO_20250521_190410.jpg,43.642099,-8.087529
67,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,0.0,0.0,PHOTO_20250521_182315.jpg,43.646536,-8.083630
118,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,95.0,0.0,0.0,PHOTO_20250521_173734.jpg,43.658095,-8.058773
119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,95.0,0.0,0.0,0.0,PHOTO_20250521_173603.jpg,43.658098,-8.058502
179,46.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_170823.jpg,43.656599,-8.053743
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2393,0.0,30.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250728_135838.jpg,42.454393,-8.872499
2395,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250728_135552.jpg,42.454834,-8.872583
2406,0.0,85.0,0.0,0.0,0.0,0.0,0.0,25.0,0.0,0.0,0.0,PHOTO_20250728_132836.jpg,42.426537,-8.655816
2439,1.0,0.0,0.0,0.0,90.0,0.0,0.0,5.0,0.0,0.0,0.0,PHOTO_20250728_115900.jpg,42.428349,-8.655235


In [6]:
"""
La primera comprobación que hacemos es mirar la columna "punto verificación", esta columna no tiene nada que ver con las columnas objetivo que tienen los porcentajes,
sin embargo es una columna numérica que está justo delante de estas columnas objetivo y por tanto es posible que por algún desplazamiento u otro problema los valores
que deberían estar en nuestros atributos se hayan quedado en la columna incorrecta.
De los registros en los que la suma no daba el valor 100, hay 6 registros que si lo sumarían si contamos el campo punto verificación, vamos a revisar si tendría sentido
desplazar por ejemplo todas sus mediciones una posición a la derecha.
Tras analizarlos, vemos que los 5 primeros registros si parecen estar desplazados por lo que vamos a sustituir sus valores por los que parecen ser correctos, sin embargo,
el último registro parece no encajar y dentro de no sumar 100 correctamente si se pueden ver los atributos en una proporción más o menos correspondiente a la que se indica.
"""
numeric_cols=["punto verificacion"]+target_cols
wrong_values[wrong_values.loc[:,numeric_cols].sum(axis=1)==100]


,punto verificacion,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos,foto,latitude,longitude
375,90.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250523_192808.jpg,43.489737,-8.166820
746,90.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250522_165855.jpg,43.455909,-8.266354
1049,80.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250524_210346.jpg,43.407698,-8.160145
1053,40.0,0.0,0.0,0.0,60.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250524_205938.jpg,43.406815,-8.160253
1089,80.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250524_205003.jpg,43.406483,-8.160653
1451,15.0,0.0,0.0,80.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250525_110835.jpg,43.317129,-8.212681


In [7]:
# Los 5 registros desplazados los sustituímos por sus valores correctos y volvemos a comprobar que registros suman un valor diferente de 100
shifted_rows=wrong_values[wrong_values.loc[:,numeric_cols].sum(axis=1)==100][:-1]
df.loc[shifted_rows.index,target_cols]=shifted_rows.iloc[:,0:10].values

# Al arreglar esas filas y volver a quedarnos con las incorrectas vemos que hay 5 menos
wrong_values=df[df.loc[:,target_cols].sum(axis=1)!=100][target_cols+["foto"]]
wrong_values

,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos,foto
6,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,70.0,PHOTO_20250521_190410.jpg
67,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,0.0,0.0,PHOTO_20250521_182315.jpg
118,0.0,0.0,0.0,0.0,0.0,0.0,0.0,95.0,0.0,0.0,PHOTO_20250521_173734.jpg
119,0.0,0.0,0.0,0.0,0.0,0.0,95.0,0.0,0.0,0.0,PHOTO_20250521_173603.jpg
179,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_170823.jpg
...,...,...,...,...,...,...,...,...,...,...,...
2393,30.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250728_135838.jpg
2395,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250728_135552.jpg
2406,85.0,0.0,0.0,0.0,0.0,0.0,25.0,0.0,0.0,0.0,PHOTO_20250728_132836.jpg
2439,0.0,0.0,0.0,90.0,0.0,0.0,5.0,0.0,0.0,0.0,PHOTO_20250728_115900.jpg


In [8]:
"""
Lo que haremos ahora es comprobar para los demás ejemplos mal etiquetados, comprobar si al menos los elementos que las etiquetas
están indicando presentes en las fotos realmente lo están y veremos si la proporción que proponen tiene sentido, es decir,
si pone que hay 80 arena y 10 alga verde, pero después la foto está llena de algas verdes y casi no hay arena pués no tendrá
sentido, por el contrario si pone 40 arena y 40 alga verde, si después en la imagen hay mitad arena mitad alga verde, es posible
que los valores reales sean 50|50 y la proporción relativa sea correcta entre ellos, otro caso que podría darse es que ponga
40 arena y 40 alga verde, pero luego en la imagen haya algo de alga roja que faltaba por identificarse.
Si lo que se da es el primer o tercer caso, lo más probable es que esos registros sean inservibles y tengamos que eliminarlos.
"""
"""
Después de una primera inspección por encima podemos sacar una conclusion, que en los casos donde solo hay un elemento visible
según la etiqueta parece que faltan otros elementos por mencionar, en la imagen se ven claramente varias cosas diferentes pero en la etiqueta
solo se recoge una de ellas con su proporción, es decir que las etiquetas están incompletas puesto que les faltan elementos que no consideran.
Por tanto, vamos a empezar por el caso de que solo se indique un elemento en la imagen.
"""
one_target=wrong_values[((wrong_values.loc[:,target_cols]>0).sum(axis=1))==1]
one_target

,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos,foto
67,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,0.0,0.0,PHOTO_20250521_182315.jpg
118,0.0,0.0,0.0,0.0,0.0,0.0,0.0,95.0,0.0,0.0,PHOTO_20250521_173734.jpg
119,0.0,0.0,0.0,0.0,0.0,0.0,95.0,0.0,0.0,0.0,PHOTO_20250521_173603.jpg
179,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_170823.jpg
235,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35.0,PHOTO_20250521_163601.jpg
387,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,PHOTO_20250523_191635.jpg
440,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250523_182443.jpg
480,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250523_180733.jpg
1411,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250525_122037.jpg
1850,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250527_124525.jpg


In [9]:
"""
La revisión de estos ejemplos es muy rápida porque tendrían que ser fotos uniformes con un único elemento presente, sin embargo,
al revisar las fotografías podemos confirmar nuestra hipótesis inicial de que las etiquetas están incompletas puesto que a simple
vista somos capaces de identificar la presencia de múltiples otros elementos que no indican.
De esta manera vamos a descartar todos los registros en los que se dé esta situación de nuestro conjunto de datos y después
revisaremos los ejemplos con porcentajes incorrectos pero varios elementos recogidos.
"""
df=df.drop(index=one_target.index)

In [10]:
mul_target=wrong_values[((wrong_values.loc[:,target_cols]>0).sum(axis=1))>1].copy()
mul_target

,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos,foto
6,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,70.0,PHOTO_20250521_190410.jpg
240,70.0,0.0,0.0,45.0,0.0,0.0,5.0,0.0,0.0,0.0,PHOTO_20250521_162844.jpg
352,40.0,0.0,0.0,20.0,0.0,0.0,20.0,0.0,0.0,0.0,PHOTO_20250523_194929.jpg
722,60.0,0.0,0.0,25.0,0.0,0.0,5.0,0.0,0.0,0.0,PHOTO_20250522_170603.jpg
758,85.0,0.0,0.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250522_165529.jpg
777,65.0,0.0,0.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250522_164804.jpg
871,0.0,0.0,0.0,5.0,0.0,0.0,0.0,90.0,0.0,0.0,PHOTO_20250522_084637.jpg
874,0.0,0.0,0.0,10.0,0.0,0.0,0.0,80.0,0.0,0.0,PHOTO_20250522_084432.jpg
897,0.0,0.0,0.0,0.0,0.0,50.0,20.0,25.0,0.0,0.0,PHOTO_20250522_083254.jpg
1066,20.0,0.0,75.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250524_205447.jpg


In [ ]:
""""
Si ahora comprobamos las imágenes de los registros que parecen tener porcentajes incompletos vemos que la mayoría de ellos
tienen sentido, los elementos que aparecen en las etiquetas están presentes en las fotografías y la proporción de estos tiene
sentido, aunque no llegue al 100% parece que si reducimos sus porcentajes proporcionalmente si se llegará a ese valor deseado,
pero hay algún ejemplo específico que tiene errores evidentes y tenemos que arreglar de forma específica:
    - La fila 2368 tiene un claro error en la columna "algas rojas" en la que tiene el valor 1111 que si eliminamos
      nos permite sumar 100 entre las columnas.

    - Hay una serie de filas que tienen valores decimales menores que 1 que parecen haber sido introducidos incorrectamente
      puesto que si movemos la coma a la derecha y en lugar de un 0,5 nos quedamos con un 5 la suma de las columnas en
      total nos daría el valor 100 que buscamos, habría dos excepciones, la fila 2271 en la que tenemos, a parte de un
      valor 0.5, un valor 70.5 que dabería ser 75 para que la suma sea correcta; la otra excepción es la fila 2242 en la que
      el valor 0.5 no debería ser un 5 sino que simplemente sobra y debería ser un 0.
      
El resto de casos deberíamos ajustarlos proporcionalmente para que la relación de valores entre columnas sea la misma pero sumen
la cantidad esperada que es 100
"""
# Cambiamos los valores incorrectos que deberían ser 0
mul_target.loc[2368,"algas pardas"] = 0
mul_target.loc[2242,"algas rojas"] = 0

# Empezamos quedándonos solo con la parte decimal y la multiplicamos por 10 y luego se la sumamos a la parte entera de cada posición
mul_target.loc[:,target_cols] = mul_target[target_cols]//1+(mul_target[target_cols]%1)*10

# Vamos a ajustar proporcionalmente las columnas, para ello primero dividimos entre la suma de cada fila y multiplicamos por 100
values = round(mul_target[target_cols].div(mul_target[target_cols].sum(axis=1),axis=0)*100,1)

# Le hemos permitido un decimal a las columnas pero queremos asegurarnos de que sumen exactamente 100 y no haya problemas de suma de 
# decimales que pueda quedar por ejemplo en un 99.9 o en un 100.1 por problemas de precisión
# Para evitar eso vamos a comprobar el residuo que falta o sobra para llegar a 100 de cada fila y se lo sumamos a la columna con el
# valor más grande puesto que será en la que menos se note la variación
residual = 100 - values.sum(axis=1)
max_col = values.idxmax(axis=1)
for i in values.index:
    values.loc[i,max_col.loc[i]] += residual.loc[i]
df.loc[mul_target.index,target_cols] = values

# Comprobamos ahora si queda algún valor incoherente
df[df.loc[:,target_cols].sum(axis=1)!=100][target_cols+["foto"]]
# Es muy importante para nuestros modelos que las clases sumen exactamente 1 al convertirlas por lo que tendrán que sumar 100 o 1

,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos,foto


In [ ]:
# Como podemos ver no queda ningún registro que tenga incoherencias en las sumas y con esto concluimos el preprocesado.
# Ahora lo que hacemos es guardar el data frame limpio con las columnas que nos interesarán
df[target_cols+["foto"]].to_csv("etiquetas_fotos.csv",index=False)

In [ ]:
"""
Vamos a estudiar la presencia de cada clase en las fotografías para comprobar si las clases están desbalanceadas y nos puede convenir darles un tratamiento diferente.
Como podemos ver que hay varias clases que aparecen en una minoría de imagenes nos interesa tener cuidado con esos elementos puesto que nuestro modelo puede aprender
a predecir siempre cero para ellos y en la mayoría de imágenes sería correcto y por tanto no sería útil a la hora de predecir dichas clases.
Podría ser buena idea aplicar data augmentation a las imágenes pero al tener tan pocas fotos con presencia de estas clases comparado con las imágenes totales al final
o la variedad no será muy buena y tendremos muchas fotos "repetidas" con pequeños cambios, o si usamos menos fotos originales estamos desaprovechando etiquetas que
tenemos disponibles
"""
(df[target_cols]>0).sum()

n. noltei             1256
z. marina               44
g. vermiculophylla     118
sedimento             1139
arena                  367
roca                    55
algas verdes           616
algas pardas           150
algas rojas             23
microfitobentos         32
dtype: int64

# Modelos

## Primer modelo - Red neuronal de multiregresión, base convolucional

In [ ]:
""""!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!"""
### Vamos a hacer la prueba de hiperparametros utilizando optuna ###
# Empezamos probando modelos relativamente pequeños y cuando decidamos cual es el mejor ya ajustamos los hiperparámetros de una version mas potente

# Primeros valores de hiperparámetros a probar
def objective_model_selection_MRConvolutionalModel(trial):
    try:
        # Modelo base a utilizar
        model_name = trial.suggest_categorical("model_name",["ConvNeXt_tiny","RegNetY_3_2GF","ResNet50","EfficientNetV2_small"])
        # Optimizador a utilizar
        optimizer = trial.suggest_categorical("optimizer", ["SGD","Adam","AdamW","RMSprop"])
        # Lr a utilizar
        lr = trial.suggest_categorical("lr",[1e-3,1e-4,1e-5])
        # Tamaños de las capas totalmente conectadas que añadimos al modelo de base (diferentes modelos rinden mejor con diferentes tamaños)
        size1 = trial.suggest_int("size1",512,1024,step=128)
        size2 = trial.suggest_int("size2",128,512,step=64)

        # Juntamos los hiperparámetros a optimizar"
        parameters = {"model_name":model_name,"optimizer":optimizer,"learning_rate":lr,"size1":size1,"size2":size2}

        print("Empezando un nuevo intento con los siguientes parámetros:",parameters)
        # Esta función se llama al final de cada época y sirve para podar las ejecuciones menos prometedoras
        def epoch_callback(loss, epoch):
            trial.report(loss, epoch)
            if trial.should_prune():
                print("El trial no era prometedor y va a ser podado")
                raise optuna.exceptions.TrialPruned()
                

        return models_training.complete_training("MRConvolutional",model_name,optimizer,train_dataloader,val_dataloader,lr1=lr,lr2=None,dropout=0.2,bloques=0,
                               size1=size1,size2=size2,patience1=5,patience2=None,max_epochs1=30,max_epochs2=None,label_smoothing=0.01,device=device,callback=epoch_callback)
    
    # Tratamiento de errores
    except optuna.exceptions.TrialPruned as e:
        raise
    except Exception as e:
        print("El trial falló, se evitarán esos hiperparámetros")
        print(e)
        raise  

# ------ Realizamos la búsqueda de hiperparámetros utilizando optuna y podando las combinaciones que no sean prometedoras
pruner = MedianPruner(n_startup_trials=15,n_warmup_steps=12)
study = optuna.create_study(
    direction="minimize",
    study_name="TFG_elegir_base_MRConvolutionalModel",
    storage="sqlite:///Hiperparametros_MRConvolutionalModel/Elegir_base_MRConvolutionalModel.db",
    load_if_exists=True,
    pruner=pruner,
)
study.optimize(objective_model_selection_MRConvolutionalModel,n_trials=70)

In [53]:
# ---------- Estudiamos los hiperparámetros con mejores resultados de la primera prueba y asi elegimos el mejor modelo base

# Mejores parámetros obtenidos en la primera prueba
study = optuna.load_study(
    study_name="TFG_elegir_base_MRConvolutionalModel",
    storage="sqlite:///Hiperparametros_MRConvolutionalModel/Elegir_base_MRConvolutionalModel.db"
)
top_trials = sorted(
    [t for t in study.trials if t.state == TrialState.COMPLETE],
    key=lambda t: t.value
)[:20]
print("Hiperparámetros de los modelos ordenados del mejor al peor:")
print("-----------------------------------------------------------------------------------------------")
for i, trial in enumerate(top_trials, 1):
    print(f"Resultado: {trial.value:.4f}{"|":^3}Duración: {str(trial.duration).split(".")[0]}{"|":^3}Parámetros: {trial.params}")


""""
Vemos claramente que la mejor base convolucional es ConvNeXt_tiny con los optimizadores Adam y AdamW entre los que podemos todavía hacer una comparativa más
profunda para elegir el mejor de los dos, el mejor learning rate con este modelo específico parece estar en torno al 0.001 pero en realidad no vamos a usar
exactamente el modelo ConvNeXt_tiny sino que probaremos una versión suya un poco más potente por lo que al learning rate le vamos a dar una búsqueda un poco
más abierta para encontrar el mejor valor, también porque como esta primera prueba erán ejecuciones más cortas, es normal que un learning rate menor funcione
mejor en esta situacioón, en cuanto al tamaño de las capas totalmente conectadas que tenemos añadirdas parece que los valores están en tornoal 1000 para la 
primra y el 300-400 para la segunda aunque no ha habido tantas pruebas con variación de tamaños por lo que la exploraremos un poco en la siguiente prueba.
Para la siguiente prueba de hiperparámetros vamos a probar dentro de la familia ConvNeXt cual sería el mejor modelo y optimizaremos los hiperparámetros de
estos para conseguir los mejores resultados, a priori parece que un modelo más potente de la misma familia debería funcionar mejor, pero es posible que sea
demasiado potente y se adapte demasiado a los ejemplos de entrenamiento generalizando mal posteriormente.
"""

Hiperparámetros de los modelos ordenados del mejor al peor:
-----------------------------------------------------------------------------------------------
Resultado: 0.1445 | Duración: 0:17:29 | Parámetros: {'model_name': 'ConvNeXt_tiny', 'optimizer': 'Adam', 'learning rate': 0.001, 'size1': 1024, 'size2': 384}
Resultado: 0.1449 | Duración: 0:18:21 | Parámetros: {'model_name': 'ConvNeXt_tiny', 'optimizer': 'AdamW', 'learning rate': 0.001, 'size1': 896, 'size2': 320}
Resultado: 0.1462 | Duración: 0:15:54 | Parámetros: {'model_name': 'ConvNeXt_tiny', 'optimizer': 'Adam', 'learning rate': 0.001, 'size1': 1024, 'size2': 448}
Resultado: 0.1466 | Duración: 0:15:45 | Parámetros: {'model_name': 'ConvNeXt_tiny', 'optimizer': 'Adam', 'learning rate': 0.001, 'size1': 1024, 'size2': 320}
Resultado: 0.1469 | Duración: 0:23:42 | Parámetros: {'model_name': 'ConvNeXt_tiny', 'optimizer': 'Adam', 'learning rate': 0.001, 'size1': 1024, 'size2': 384}
Resultado: 0.1472 | Duración: 0:19:00 | Parámetros: {'

'"\nVemos claramente que la mejor base convolucional es ConvNeXt_tiny con los optimizadores Adam y AdamW entre los que podemos todavía hacer una comparativa más\nprofunda para elegir el mejor de los dos, el mejor learning rate con este modelo específico parece estar en torno al 0.001 pero en realidad no vamos a usar\nexactamente el modelo ConvNeXt_tiny sino que probaremos una versión suya un poco más potente por lo que al learning rate le vamos a dar una búsqueda un poco\nmás abierta para encontrar el mejor valor, también porque como esta primera prueba erán ejecuciones más cortas, es normal que un learning rate menor funcione\nmejor en esta situacioón, en cuanto al tamaño de las capas totalmente conectadas que tenemos añadirdas parece que los valores están en tornoal 1000 para la \nprimra y el 300-400 para la segunda aunque no ha habido tantas pruebas con variación de tamaños por lo que la exploraremos un poco en la siguiente prueba.\nPara la siguiente prueba de hiperparámetros vamos 

In [ ]:
""""!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!"""
### Vamos a hacer la segunda prueba de hiperparametros utilizando optuna ###

# Valores detallados de hiperparámetros a probar
def objective_hiperparams_MRConvolutionalModel(trial):
    try:
        # Modelo base a utilizar
        model_name = trial.suggest_categorical("model_name",["ConvNeXt_tiny","ConvNeXt_small"])
        # Optimizador a utilizar
        optimizer = trial.suggest_categorical("optimizer", ["Adam","AdamW"])
        # Bloques del modelo base que vamos a descongelar durante el fine tuning (0 significa que no habrá fine tuning)
        bloques = trial.suggest_int("bloques",0,3)
        # Lr a utilizar tanto en la primera parte como durante el fine tuning
        lr1 = trial.suggest_float("lr1",1e-4,1e-2,log=True)
        lr2 = trial.suggest_float("lr2",1e-6,1e-4,log=True) if bloques>0 else None
        # Tamaños de las capas totalmente conectadas que añadimos al modelo de base (diferentes modelos rinden mejor con diferentes tamaños)
        size1 = trial.suggest_int("size1",512,1280,step=256)
        size2 = trial.suggest_int("size2",128,512,step=128)
        # Porcentaje de neuronas que se desactivan en el dropout
        dropout = trial.suggest_float("dropout", 0.1, 0.3)
        # Número máximo de épocas para cada etapa (nos ayudará a entender si alguna de las etapas sería mejor evitarla o que dure poco)
        max_epochs1 = trial.suggest_int("max_epochs1",1,40)
        max_epochs2 = trial.suggest_int("max_epochs2",1,50-max_epochs1) if bloques>0 else None
        # Valor del label smoothing que servirá para determinar el valor a repartir entre las clases y evitar así los 0s
        label_smoothing = trial.suggest_categorical("label_smoothing",[0.1,0.01,0.001])

        # Juntamos los hiperparámetros a optimizar
        parameters = {"model_name":model_name,"optimizer":optimizer,"bloques":bloques,"lr1":lr1,"lr2":lr2,"size1":size1,"size2":size2,"dropout":dropout,
                      "max_epochs1":max_epochs1,"max_epochs2":max_epochs2,"label_smoothing":label_smoothing}

        print("Empezando un nuevo intento con los siguientes parámetros:",parameters)
        # Esta función se llama al final de cada época y sirve para podar las ejecuciones menos prometedoras
        def epoch_callback(loss, epoch):
            trial.report(loss, epoch)
            if trial.should_prune():
                print("El trial no era prometedor y va a ser podado")
                raise optuna.exceptions.TrialPruned()

        return models_training.complete_training("MRConvolutional",model_name,optimizer,train_dataloader,val_dataloader,lr1=lr1,lr2=lr2,dropout=dropout,bloques=bloques,size1=size1,
                        size2=size2,patience1=10,patience2=10,max_epochs1=max_epochs1,max_epochs2=max_epochs2,label_smoothing=label_smoothing,device=device,callback=epoch_callback)
    
    # Tratamiento de errores
    except optuna.exceptions.TrialPruned as e:
        raise
    except Exception as e:
        print("El trial falló, se evitarán esos hiperparámetros")
        print(e)
        raise  

# ------ Realizamos la búsqueda de hiperparámetros utilizando optuna y podando las combinaciones que no sean prometedoras
pruner = MedianPruner(n_startup_trials=15,n_warmup_steps=15)
study = optuna.create_study(
    direction="minimize",
    study_name="TFG_hiperparametros_MRConvolutionalModel",
    storage="sqlite:///Hiperparametros_MRConvolutionalModel/Hiperparametros_MRConvolutionalModel.db",
    load_if_exists=True,
    pruner=pruner,
)
study.optimize(objective_hiperparams_MRConvolutionalModel,n_trials=70)

In [ ]:
# ---------- Estudiamos los hiperparámetros con mejores resultados de la segunda prueba

# Mejores parámetros obtenidos en la segunda prueba
study = optuna.load_study(
    study_name="TFG_hiperparametros_MRConvolutionalModel",
    storage="sqlite:///Hiperparametros_MRConvolutionalModel/Hiperparametros_MRConvolutionalModel.db"
)
top_trials = sorted(
    [t for t in study.trials if t.state == TrialState.COMPLETE],
    key=lambda t: t.value
)[:20]
print("Hiperparámetros de los modelos ordenados del mejor al peor:")
print("-----------------------------------------------------------------------------------------------")
for i, trial in enumerate(top_trials, 1):
    print(f"Resultado: {trial.value:.4f}{"|":^3}Duración: {str(trial.duration).split(".")[0]}{"|":^3}Parámetros: {trial.params}")
    
"""
En base a los resultados se ve como el modelo ConvNeXt_small es el que mejor valores obtiene tal y como sospechábamos, en este punto podríamos
probar con el siguiente tamaño de modelo de esta misma familia para comprobar si se obtiene una mejora considerable, pero este modelo small ya
estaba dando algunos problemas de memoria por lo que para la comparativa de modelo nos quedaremos con el modelo small pero queda abierto probar
modelos superiores para conseguir posiblemente mejores resultados, aun asi cabe destacar que el modelo small ya esta muy cerca del valor 0 en
la convergencia kl de entrenamiento por lo que a menos que se utilizarán etiquetas más complejas parece que es lo bastante potente.
Para el resto de hiperparámetros nos quedaremos con los que mejor resultados nos han dado empezando por el optimizador AdamW, un lr1 que este
entre 2e-4 y 8e-4, un lr2 que este entre 4e-5 y 5e-5, los tamaños de las capas totalmente conectadas parecen funcionar mejor con los valores
1024 y 512, un dropout reducido entre 0.1 y 0.15, un label smoothing de 0.01, y después, las épocas que no nos indican que ninguna de las dos
etapas sea innecesaria puesto que el número de épocas se mantiene más o menos proporcional. Por último a a la hora de decidir cuantos bloques
descongelar durante el fine tuning, vemos que los resultados no cambian mucho pero el tiempo de entrenamiento si aumenta más notablemente,
aunque es verdad que depende en parte del número de épocas, eñ aumento de duración parece más notable que la mejoría de resultados y teniendo
en cuenta que el entrenamiento final será más etapas y con más ejemplos, parece mejor idea descongelar solo dos bloques.
"""

Hiperparámetros de los modelos ordenados del mejor al peor:
-----------------------------------------------------------------------------------------------
Resultado: 0.1824 | Duración: 0:58:29 | Parámetros: {'model_name': 'ConvNeXt_small', 'optimizer': 'AdamW', 'bloques': 3, 'lr1': 0.0008280567061290827, 'lr2': 4.0280443576384534e-05, 'size1': 1024, 'size2': 512, 'dropout': 0.12474585347096058, 'max_epochs1': 29, 'max_epochs2': 14, 'label_smoothing': 0.01}
Resultado: 0.1825 | Duración: 0:47:15 | Parámetros: {'model_name': 'ConvNeXt_small', 'optimizer': 'AdamW', 'bloques': 3, 'lr1': 0.0002609393835081019, 'lr2': 3.9757527648028154e-05, 'size1': 1024, 'size2': 512, 'dropout': 0.13756081847341975, 'max_epochs1': 25, 'max_epochs2': 19, 'label_smoothing': 0.01}
Resultado: 0.1841 | Duración: 0:40:35 | Parámetros: {'model_name': 'ConvNeXt_small', 'optimizer': 'AdamW', 'bloques': 2, 'lr1': 0.00022409696827783895, 'lr2': 5.111934977799174e-05, 'size1': 1024, 'size2': 512, 'dropout': 0.19456802

'\nEn base a los resultados se ve como el modelo ConvNeXt_small es el que mejor valores obtiene tal y como sospechábamos, en este punto podríamos\nprobar con el siguiente tamaño de modelo de esta misma familia para comprobar si se obtiene una mejora considerable, pero este modelo small ya\nestaba dando algunos problemas de memoria por lo que para la comparativa de modelo nos quedaremos con el aunque queda abierta\n'

## Segundo modelo - Red neuronal de multiregresión, base convolucional con capa de histograma

## Tercer modelo - Red neuronal de multiregresión, base vision transformer

# Rendimiento en conjunto de test

In [6]:
# Elegimos la gpu si está disponible y si no la cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cargamos en data frames los conjuntos de datos ya divididos
test = pd.read_csv("./dataset_dividido/test.csv")

# Creamos el datasets y dataloader que vamos a usar para comparar los modelos ya entrenados
test_dataset = CustomImageDataset("./fotos_recortadas",test,True)
test_dataloader = DataLoader(test_dataset,batch_size=128,shuffle=False,pin_memory=True)

modelMRConv = MRConvolutionalModel("ConvNeXt_tiny",0.15,size1=512,size2=128).to(device)
modelMRConv_Hist = MRConvolutionalModelHistogram("ConvNeXt_tiny",22,0.15,size1=1024,size2=512,size3=128).to(device)

modelMRConv.load_state_dict(torch.load("MRConvolutional_ConvNeXt_tiny_best_model.pth",map_location=device))
modelMRConv_Hist.load_state_dict(torch.load("./MRConvolutional_Histogram_ConvNeXt_tiny_best_model.pth",map_location=device))

<All keys matched successfully>

In [7]:
mae_loss_module = nn.L1Loss()
kl_divergence_module = nn.KLDivLoss(reduction="batchmean")

val_mae_loss = 0
val_kl_divergence = 0
class_mae_loss = 0

val_mae_loss_Hist = 0
val_kl_divergence_Hist = 0
class_mae_loss_Hist = 0

total_samples = 0

with torch.no_grad():
    modelMRConv.eval()
    modelMRConv_Hist.eval()
    for data_inputs, data_labels in test_dataloader:
        data_inputs = data_inputs.to(device)
        data_labels = data_labels.to(device)
        total_samples += data_inputs.size(0)

        # Vamos a evitar que las etiquetas tengan el valor exacto 0 para evitar que la divergencia kl colapse
        smooth_labels = data_labels*(1-0.01)+0.01/10
        log_preds = modelMRConv(data_inputs)
        log_preds_Hist = modelMRConv_Hist(data_inputs)

        # Calcular el valor de la función de pérdida mae y la divergencia kl
        y_pred = torch.exp(log_preds)
        y_pred_Hist = torch.exp(log_preds_Hist)

        # Calculamos las pérdidas
        val_mae_loss += mae_loss_module(y_pred,data_labels).item()
        val_kl_divergence += kl_divergence_module(log_preds,smooth_labels).item()

        val_mae_loss_Hist += mae_loss_module(y_pred_Hist,data_labels).item()
        val_kl_divergence_Hist += kl_divergence_module(log_preds_Hist,smooth_labels).item()

        # Calculamos ademas el MAE por cada clase individual
        class_mae_loss += torch.mean(torch.abs(y_pred-data_labels),dim=0)
        class_mae_loss_Hist += torch.mean(torch.abs(y_pred_Hist-data_labels),dim=0)

# Construimos una estructura del MAE por clase para imprimirlo
class_mae = " | ".join([str(round(i,4)) for i in (class_mae_loss/len(test_dataloader) ).tolist()])
class_mae_Hist = " | ".join([str(round(i,4)) for i in (class_mae_loss_Hist/len(test_dataloader)).tolist()])

# Imprimimos resultados para comparar
print("Modelo de red multiregresión con base convolucional SIN capa de Histograma.\n"
      "Divergencia KL: %.4f , MAE Loss: %.4f\n"
      "MAE por clases: {%s}\n"% 
     (val_kl_divergence/len(test_dataloader),val_mae_loss/len(test_dataloader),class_mae))
print("Modelo de red multiregresión con base convolucional CON capa de Histograma.\n"
      "Divergencia KL: %.4f , MAE Loss: %.4f\n"
      "MAE por clases: {%s}"% 
     (val_kl_divergence_Hist/len(test_dataloader),val_mae_loss_Hist/len(test_dataloader),class_mae_Hist))

Modelo de red multiregresión con base convolucional SIN capa de Histograma.
Divergencia KL: 0.3065 , MAE Loss: 0.0418
MAE por clases: {0.0785 | 0.0062 | 0.0236 | 0.1267 | 0.0605 | 0.0086 | 0.0787 | 0.0153 | 0.0071 | 0.0125}

Modelo de red multiregresión con base convolucional CON capa de Histograma.
Divergencia KL: 0.3460 , MAE Loss: 0.0462
MAE por clases: {0.0767 | 0.0096 | 0.0251 | 0.1539 | 0.0733 | 0.0119 | 0.0755 | 0.0145 | 0.0082 | 0.0129}
